### Using DTW to match precipitaiton events to the consequent recharge event. 

In [1]:
"""Pull recharge events for 10 random NE wells using DTW-based matching."""

import numpy as np
import pandas as pd
import geopandas as gpd
import os
from dtw import dtw
import matplotlib.pyplot as plt


Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



#### Loading in the data. Code pulled from Groundwater_x_Precipitation_FIXED.ipynb

In [2]:
well_sites = r"C:\Users\romin\OneDrive\Groundwater\RpSy Data\Site information for all selected wells.xlsx"
state_boundaries = r"C:\Users\romin\OneDrive\Groundwater\cb_2022_us_state_500k"
recharge_dir = r"C:/Users/romin/OneDrive/Groundwater/RpSy Data"
daymet_dir = "./daymet"

NE_states = [
    "Connecticut", "Maine", "Massachusetts", "New Hampshire",
    "Rhode Island", "Vermont", "New Jersey", "New York", "Pennsylvania",
]

# ============================================================
# Load wells and filter to NE states
# ============================================================

def load_sites(path=well_sites):
    df = pd.read_excel(path)
    df = df.rename(columns={
        "ID": "usgs_id",
        "Lat": "lat",
        "Long": "lon",
        "depth (m)": "depth",
    })
    return df

def select_ne_wells(df_sites, states_shp=state_boundaries):
    states = gpd.read_file(states_shp)
    ne = states[states["NAME"].isin(NE_states)].set_crs(epsg=4326, allow_override=True)
    gdf_sites = gpd.GeoDataFrame(
        df_sites,
        geometry=gpd.points_from_xy(df_sites["lon"], df_sites["lat"]),
        crs="EPSG:4326",
    )
    joined = gpd.sjoin(gdf_sites, ne[["NAME", "geometry"]], how="inner", predicate="within")
    joined = joined.rename(columns={"NAME": "state"}).drop(columns=["index_right"])
    return joined

df_sites = load_sites()
df_ne_wells = select_ne_wells(df_sites)


#### Determining the start and end date of RpSy data.

In [3]:
def get_start_end_dates(df, date_col="Date"):
    df[date_col] = pd.to_datetime(df[date_col])
    return df[date_col].min(), df[date_col].max() 

start_dates = []
end_dates = []
for wid in df_ne_wells["usgs_id"]:
    df = pd.read_csv(f"{recharge_dir}/{wid}.csv")
    start_date, end_date = get_start_end_dates(df)
    start_dates.append(start_date)
    end_dates.append(end_date)

df_ne_wells["start_date"] = start_dates
df_ne_wells["end_date"] = end_dates
df_ne_wells["record_length"] = (df_ne_wells["end_date"] - df_ne_wells["start_date"]).dt.days / 365.25

#### Filtering for wells with a record length >= 5 years.

In [4]:
eligible_wells = df_ne_wells[df_ne_wells["record_length"] >= 5].copy()
print(f"Wells with >=5 year records: {len(eligible_wells)}")

selected_wells = eligible_wells.sample(n=10, random_state=42)
print(selected_wells[["usgs_id", "record_length", "start_date", "end_date"]])

Wells with >=5 year records: 163
             usgs_id  record_length start_date   end_date
426  425803077151201      18.995209 2003-10-02 2022-09-30
403  421746074180201      15.994524 2006-10-02 2022-09-30
422  424520070562401      13.993155 1985-10-02 1999-09-30
306  404639074230001      12.993840 2009-10-02 2022-09-30
356  414330076280501      23.994524 1998-10-02 2022-09-30
274  400229075104601       9.993155 2012-10-02 2022-09-30
458  444904074455201      19.994524 2002-10-02 2022-09-30
302  404140077354001      16.996578 1999-10-02 2016-09-30
364  415228070554601       7.994524 2000-10-02 2008-09-30
439  434217073010601       5.993155 2016-10-02 2022-09-30


#### Functions to assist the DTW package and the calculated Precipitation/Recharge event table. Written by Professor David Litwin. 

In [5]:
# DTW helper functions
def get_events(values, threshold):
    """Return start/end index pairs for runs of consecutive values > threshold."""
    above = (values > threshold).astype(int)
    padded = np.concatenate(([0], above, [0]))
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0] - 1
    return starts, ends

def precip_recharge_event_table(df, precip_col, recharge_col, alignment, precip_threshold=0.0):
    """Return a table of precipitation and recharge events based on DTW alignment."""
    dates = df.index
    precip_vals = df[precip_col].values
    recharge_vals = df[recharge_col].values
    starts, ends = get_events(precip_vals, precip_threshold)

    rows = []
    prev_recharge_idx = None
    for s, e in zip(starts, ends):
        precip_idx = np.arange(s, e + 1)
        mask = np.isin(alignment.index2, precip_idx)
        recharge_idx = np.unique(alignment.index1[mask])

        if prev_recharge_idx is not None and recharge_idx.size > 0:
            test = recharge_idx > prev_recharge_idx.max()
            recharge_idx = recharge_idx[test]   # dropped silently, no print

        row = {
            "precip_start_date": dates[s],
            "precip_end_date": dates[e],
            "precip_total": precip_vals[precip_idx].sum(),
            "precip_peak": precip_vals[precip_idx].max(),
        }
        if recharge_idx.size == 0:
            row.update({
                "recharge_start_date": pd.NaT,
                "recharge_end_date": pd.NaT,
                "recharge_total": np.nan,
                "recharge_peak": np.nan,
                "n_recharge_days": 0,
            })
        else:
            row.update({
                "recharge_start_date": dates[recharge_idx.min()],
                "recharge_end_date": dates[recharge_idx.max()],
                "recharge_total": recharge_vals[recharge_idx].sum(),
                "recharge_peak": recharge_vals[recharge_idx].max(),
                "n_recharge_days": recharge_idx.size,
            })
        rows.append(row)
        prev_recharge_idx = recharge_idx if recharge_idx.size > 0 else prev_recharge_idx

    return pd.DataFrame(rows)

def check_overlapping_recharge_events(event_table):
    """Check for overlapping recharge events in the event table."""
    sorted_table = event_table.sort_values("recharge_start_date")
    overlaps = []
    for i in range(len(sorted_table) - 1):
        current_end = sorted_table.iloc[i]["recharge_end_date"]
        next_start = sorted_table.iloc[i + 1]["recharge_start_date"]
        if pd.notna(current_end) and pd.notna(next_start) and current_end >= next_start:
            overlap_days = (current_end - next_start).days + 1
            overlaps.append((i, current_end, next_start, overlap_days))
    return overlaps

#### Running the DTW function for the 163 eligible wells. 

In [6]:
os.makedirs("dtw_event_tables", exist_ok=True)

eligible_wells = df_ne_wells[df_ne_wells["record_length"] >= 5].copy()
print(f"Total wells with >=5 year records: {len(eligible_wells)}")

def run_dtw_pipeline(site_id, recharge_dir=recharge_dir, daymet_dir=daymet_dir,
                      out_dir="dtw_event_tables",
                      precip_threshold=2.5, min_lag=0, max_lag=5):
    recharge_path = f"{recharge_dir}/{site_id}.csv"
    precip_path = f"{daymet_dir}/{site_id}.csv"

    if not os.path.exists(recharge_path) or not os.path.exists(precip_path):
        return None, "missing file"

    df_recharge = pd.read_csv(recharge_path)
    df_precip = pd.read_csv(precip_path)

    df_precip["Date"] = pd.to_datetime(df_precip['year'] * 1000 + df_precip['yday'], format='%Y%j')
    df_recharge["Date"] = pd.to_datetime(df_recharge["Date"])
    df_recharge.set_index("Date", inplace=True)
    df_precip.set_index("Date", inplace=True)

    df = pd.merge(df_recharge, df_precip, on="Date", how="inner")
    df.drop(columns=["year", "yday"], inplace=True, errors="ignore")

    if len(df) < 30:
        return None, "too short"

    recharge = df["RpSy (m)"].values
    precip = df["prcp (mm/day)"].values

    if np.std(recharge) == 0 or np.std(precip) == 0:
        return None, "zero variance"

    recharge_norm = recharge / np.std(recharge)
    precip_norm = precip / np.std(precip)

    def causal_window(iw, jw, query_size, reference_size, min_lag=min_lag, max_lag=max_lag, **kwargs):
        lag = iw - jw
        ok = lag >= min_lag
        if max_lag is not None:
            ok = ok & (lag <= max_lag)
        return ok

    try:
        alignment = dtw(
            recharge_norm, precip_norm,
            step_pattern="symmetric2",
            window_type=causal_window,
            window_args={"min_lag": min_lag, "max_lag": max_lag},
            keep_internals=True,
            open_begin=False,
            open_end=False,
        )
    except Exception as e:
        return None, f"dtw failed: {e}"

    event_table = precip_recharge_event_table(
        df, "prcp (mm/day)", "RpSy (m)", alignment, precip_threshold=precip_threshold
    )

    out_path = f"{out_dir}/{site_id}_dtw_event_table.csv"
    event_table.to_csv(out_path, index=False)

    return event_table, "ok"


all_event_tables = {}
failed_wells = {}

for i, site_id in enumerate(eligible_wells["usgs_id"]):
    site_id = str(site_id)
    print(f"\rProcessing well {i+1}/{len(eligible_wells)}...", end="", flush=True)

    out_path = f"dtw_event_tables/{site_id}_dtw_event_table.csv"
    if os.path.exists(out_path):
        all_event_tables[site_id] = pd.read_csv(out_path)
        continue

    table, status = run_dtw_pipeline(site_id)
    if table is not None:
        all_event_tables[site_id] = table
    else:
        failed_wells[site_id] = status

print(f"\n\nCompleted: {len(all_event_tables)} of {len(eligible_wells)} wells")
print(f"Failed/skipped: {len(failed_wells)}")
if failed_wells:
    from collections import Counter
    reasons = Counter(failed_wells.values())
    print("Failure reasons:", dict(reasons))

Total wells with >=5 year records: 163
Processing well 163/163...

Completed: 163 of 163 wells
Failed/skipped: 0


#### Time series visualization

In [11]:
import matplotlib
matplotlib.use('QtAgg')
import matplotlib.pyplot as plt

def plot_dtw_events(site_id, event_table, recharge_dir=recharge_dir, daymet_dir=daymet_dir):
    df_recharge = pd.read_csv(f"{recharge_dir}/{site_id}.csv")
    df_precip = pd.read_csv(f"{daymet_dir}/{site_id}.csv")
    df_precip["Date"] = pd.to_datetime(df_precip['year'] * 1000 + df_precip['yday'], format='%Y%j')
    df_recharge["Date"] = pd.to_datetime(df_recharge["Date"])
    df_recharge.set_index("Date", inplace=True)
    df_precip.set_index("Date", inplace=True)
    df = pd.merge(df_recharge, df_precip, on="Date", how="inner")
    df.drop(columns=["year", "yday"], inplace=True, errors="ignore")

    date_cols = ["recharge_start_date", "recharge_end_date", "precip_start_date", "precip_end_date"]
    event_table[date_cols] = event_table[date_cols].apply(pd.to_datetime)

    fig, ax1 = plt.subplots(figsize=(10, 5))
    color = 'tab:red'
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Recharge (m)', color=color)
    recharge_max = df["RpSy (m)"].max()
    ax1.set_ylim(0, recharge_max * 1.5)
    ax1.plot(df.index, df["RpSy (m)"], color=color)
    ax1.tick_params(axis='y', labelcolor=color)
   
    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Precipitation (mm)', color=color)
    ax2.plot(df.index, df["prcp (mm/day)"], color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    precip_max = df["prcp (mm/day)"].max()
    ax2.set_ylim(precip_max * 1.5, 0)
    axmax = ax1.get_ylim()[1]
    for _, row in event_table.iterrows():
        if pd.notna(row["recharge_start_date"]) and pd.notna(row["recharge_end_date"]):
            x = [row["recharge_end_date"], row["recharge_start_date"],
                 row["precip_start_date"], row["precip_end_date"]]
            y = [0, 0, axmax, axmax]
            ax1.fill(x, y, color='gray', alpha=0.2)
    ax1.set_title(f"Well {site_id}")
    plt.show()


In [36]:
preview_ids = list(all_event_tables.keys())[:2]

for site_id in preview_ids:
    plot_dtw_events(site_id, all_event_tables[site_id])

#### Adding the covariates as defines in Groundwater_x_Precipiation_FIXED.ipynb.

In [23]:
def add_covariates(event_table):
    """
    Adds DUR, AVG, RPR to a DTW-based event table.
    MAG uses precip_total directly (from the DTW event boundaries).
    RECH uses recharge_total directly (raw RpSy, per current guidance).
    """
    df = event_table.copy()

    date_cols = ["precip_start_date", "precip_end_date", "recharge_start_date", "recharge_end_date"]
    df[date_cols] = df[date_cols].apply(pd.to_datetime)

    # DUR: days spanned by the precip event (start to end, inclusive)
    df["DUR"] = (df["precip_end_date"] - df["precip_start_date"]).dt.days + 1
  
    # MAG: using the full precip total from this event's DTW-derived window
    df["MAG"] = df["precip_total"]
    
    # AVG: average daily rate across the event
    df["AVG"] = df["MAG"] / df["DUR"]
   
    # RECH: raw recharge total (Sy not yet applied, per current guidance)
    df["RECH"] = df["recharge_total"]
   
    # RPR: recharge (converted m -> mm) / precip magnitude (mm)
    df["RPR"] = (df["RECH"] * 1000) / df["MAG"]
    return df

#### Grouping events by season and plotting the covariates. 

In [24]:
def add_season(event_table, date_col="precip_start_date"):
    """
    Adds a season column based on meteorological seasons:
    DJF = Winter, MAM = Spring, JJA = Summer, SON = Fall
    """
    df = event_table.copy()
    month = df[date_col].dt.month

    season_map = {
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Spring", 4: "Spring", 5: "Spring",
        6: "Summer", 7: "Summer", 8: "Summer",
        9: "Fall", 10: "Fall", 11: "Fall",
    }
    df["season"] = month.map(season_map)
    return df


def plot_rpr_by_season(event_table, well_id, x_col="DUR", x_label="Duration (days)"):
    """
    Four separate panels, one per season, each showing RPR vs a
    chosen storm characteristic with its own trend line.
    """
    df = event_table.dropna(subset=[x_col, "RPR"]).copy()

    seasons = ["Winter", "Spring", "Summer", "Fall"]
    colors = {"Winter": "tab:blue", "Spring": "tab:green",
              "Summer": "tab:orange", "Fall": "tab:brown"}

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    for ax, season in zip(axes.flat, seasons):
        season_data = df[df["season"] == season]
        color = colors[season]

        ax.scatter(season_data[x_col], season_data["RPR"],
                   alpha=0.5, s=25, color=color)

        if len(season_data) > 2:
            coeffs = np.polyfit(season_data[x_col], season_data["RPR"], 1)
            x_line = np.linspace(season_data[x_col].min(), season_data[x_col].max(), 50)
            y_line = coeffs[0] * x_line + coeffs[1]
            ax.plot(x_line, y_line, color="red", linewidth=2,
                    label=f"slope: {coeffs[0]:.3f}")
            ax.legend(fontsize=8)

        ax.set_xlabel(x_label)
        ax.set_ylabel("RPR")
        ax.set_title(f"{season} (n={len(season_data)})")
        ax.grid(alpha=0.3)

    fig.suptitle(f"Well {well_id} — RPR vs {x_label}, by Season", y=1.00)
    fig.tight_layout()
    plt.show()

In [25]:
site_id = list(all_event_tables.keys())[0]
enriched_event_table = add_covariates(all_event_tables[site_id])
enriched_event_table = add_season(enriched_event_table)

print(enriched_event_table["season"].value_counts())



season
Summer    213
Winter    200
Spring    198
Fall      174
Name: count, dtype: int64


In [26]:
plot_rpr_by_season(enriched_event_table, site_id, x_col="DUR", x_label="Duration (days)")
plot_rpr_by_season(enriched_event_table, site_id, x_col="MAG", x_label="Magnitude (mm)")
plot_rpr_by_season(enriched_event_table, site_id, x_col="AVG", x_label="Average rate (mm/day)")

#### Cumulative plot of the precipiation/recharge relationship. A linear line indicates that all events are equally responsible for recharge, where as a slightly convex line suggests larger events are disproportionately responsible for recharge. 

In [27]:
def plot_cumulative_precip_recharge(event_table, well_id, season=None,
                                      precip_col="MAG", recharge_col="RECH"):
    """
    Sorts events from largest to smallest precipitation, then plots
    the cumulative fraction of total precipitation (x) against the
    cumulative fraction of total recharge (y).
    If season is given, filters the event_table to just that season first.
    """
    df = event_table.dropna(subset=[precip_col, recharge_col]).copy()

    if season is not None:
        df = df[df["season"] == season]

    if len(df) == 0:
        print(f"No events found for {well_id} in season={season}. Skipping.")
        return

    df = df.sort_values(precip_col, ascending=False).reset_index(drop=True)

    total_precip = df[precip_col].sum()
    total_recharge = df[recharge_col].sum()

    df["cum_precip"] = df[precip_col].cumsum()
    df["cum_recharge"] = df[recharge_col].cumsum()

    df["cum_precip_frac"] = df["cum_precip"] / total_precip
    df["cum_recharge_frac"] = df["cum_recharge"] / total_recharge

    x = np.concatenate(([0], df["cum_precip_frac"].values))
    y = np.concatenate(([0], df["cum_recharge_frac"].values))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(x, y, marker="o", color="black", markersize=4)
    

    ax.set_xlabel("Cumulative fraction of total precipitation")
    ax.set_ylabel("Cumulative fraction of total recharge")
    title = f"Well {well_id}"
    if season is not None:
        title += f" — {season} (n={len(df)})"
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


def plot_cumulative_by_season(event_table, well_id, precip_col="MAG", recharge_col="RECH"):
    
    for season in ["Winter", "Spring", "Summer", "Fall"]:
        plot_cumulative_precip_recharge(event_table, well_id, season=season,
                                          precip_col=precip_col, recharge_col=recharge_col)

In [28]:
plot_cumulative_by_season(enriched_event_table, site_id)

C:\Users\romin\AppData\Local\Temp\ipykernel_10036\2972334982.py:42: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()
C:\Users\romin\AppData\Local\Temp\ipykernel_10036\2972334982.py:42: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()
C:\Users\romin\AppData\Local\Temp\ipykernel_10036\2972334982.py:42: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()
C:\Users\romin\AppData\Local\Temp\ipykernel_10036\2972334982.py:42: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.l

#### Ploting the annual trends in precipitation and recharge. A seasonal version is included as well. 

In [29]:
def plot_annual_trends(event_table, well_id, season=None,
                         precip_col="MAG", recharge_col="RECH", date_col="precip_start_date"):
    """
    Two stacked panels: annual total precipitation (top) and annual
    total recharge (bottom), each with a fitted linear trend line.
    """
    df = event_table.dropna(subset=[precip_col, recharge_col]).copy()

    if season is not None:
        df = df[df["season"] == season]

    if len(df) == 0:
        print(f"No events found for {well_id}, season={season}. Skipping.")
        return

    df["year"] = df[date_col].dt.year
    yearly = df.groupby("year").agg(
        precip_total=(precip_col, "sum"),
        recharge_total=(recharge_col, "sum"),
    ).reset_index()
    yearly["recharge_mm"] = yearly["recharge_total"] * 1000  # convert to mm

    if len(yearly) < 4:
        print(f"Only {len(yearly)} years of data -- too short for a meaningful trend. Skipping.")
        return

    precip_slope, precip_intercept = np.polyfit(yearly["year"], yearly["precip_total"], 1)
    recharge_slope, recharge_intercept = np.polyfit(yearly["year"], yearly["recharge_mm"], 1)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 9), sharex=True)

    ax1.bar(yearly["year"], yearly["precip_total"], color="tab:blue", alpha=0.6)
    ax1.plot(yearly["year"], precip_slope * yearly["year"] + precip_intercept,
              color="darkblue", linewidth=2, label=f"Trend: {precip_slope:+.2f} mm/yr")
    ax1.set_ylabel("Annual precipitation (mm)")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.bar(yearly["year"], yearly["recharge_mm"], color="tab:green", alpha=0.6)
    ax2.plot(yearly["year"], recharge_slope * yearly["year"] + recharge_intercept,
              color="darkgreen", linewidth=2, label=f"Trend: {recharge_slope:+.2f} mm/yr")
    ax2.set_ylabel("Annual recharge (mm)")
    ax2.set_xlabel("Year")
    ax2.legend()
    ax2.grid(alpha=0.3)

    title = f"Well {well_id} — Annual Precipitation and Recharge Trends"
    if season is not None:
        title += f" ({season})"
    fig.suptitle(title, y=0.98)
    fig.tight_layout()
    plt.show()


def plot_annual_trends_season(event_table, well_id, precip_col="MAG", recharge_col="RECH",
                                   date_col="precip_start_date"):
    """
    One figure, 4 columns (Winter, Spring, Summer, Fall) x 2 rows
    (precip on top, recharge on bottom), each panel with its own
    linear trend line.
    """
    seasons = ["Winter", "Spring", "Summer", "Fall"]
    fig, axes = plt.subplots(2, 4, figsize=(20, 9), sharey="row")

    for col, season in enumerate(seasons):
        df = event_table.dropna(subset=[precip_col, recharge_col]).copy()
        df = df[df["season"] == season]

        ax_p = axes[0, col]
        ax_r = axes[1, col]

        if len(df) == 0:
            ax_p.set_title(f"{season} (no data)")
            continue

        df["year"] = df[date_col].dt.year
        yearly = df.groupby("year").agg(
            precip_total=(precip_col, "sum"),
            recharge_total=(recharge_col, "sum"),
        ).reset_index()
        yearly["recharge_mm"] = yearly["recharge_total"] * 1000

        if len(yearly) < 4:
            ax_p.set_title(f"{season} (only {len(yearly)} yrs)")
            ax_p.bar(yearly["year"], yearly["precip_total"], color="tab:blue", alpha=0.6)
            ax_r.bar(yearly["year"], yearly["recharge_mm"], color="tab:green", alpha=0.6)
            continue

        precip_slope, precip_intercept = np.polyfit(yearly["year"], yearly["precip_total"], 1)
        recharge_slope, recharge_intercept = np.polyfit(yearly["year"], yearly["recharge_mm"], 1)

        ax_p.bar(yearly["year"], yearly["precip_total"], color="tab:blue", alpha=0.6)
        ax_p.plot(yearly["year"], precip_slope * yearly["year"] + precip_intercept,
                   color="darkblue", linewidth=2)
        ax_p.set_title(f"{season}\nslope: {precip_slope:+.2f} mm/yr")
        ax_p.grid(alpha=0.3)

        ax_r.bar(yearly["year"], yearly["recharge_mm"], color="tab:green", alpha=0.6)
        ax_r.plot(yearly["year"], recharge_slope * yearly["year"] + recharge_intercept,
                   color="darkgreen", linewidth=2)
        ax_r.set_title(f"slope: {recharge_slope:+.2f} mm/yr")
        ax_r.set_xlabel("Year")
        ax_r.grid(alpha=0.3)

    axes[0, 0].set_ylabel("Annual precipitation (mm)")
    axes[1, 0].set_ylabel("Annual recharge (mm)")

    fig.suptitle(f"Well {well_id} — Seasonal Precipitation and Recharge Trends", y=1.0)
    fig.tight_layout()
    plt.show()

In [30]:
site_id = list(all_event_tables.keys())[27]
enriched_event_table = add_covariates(all_event_tables[site_id])
enriched_event_table = add_season(enriched_event_table)

plot_annual_trends(enriched_event_table, site_id)
plot_annual_trends_season(enriched_event_table, site_id)

In [31]:
import pymannkendall as mk

def compute_well_trend(event_table, value_col, date_col="precip_start_date"):
    """
    Runs Mann-Kendall on annual totals for one well. Returns slope
    and significance, or None if too little data.
    """
    df = event_table.dropna(subset=[value_col]).copy()
    df["year"] = df[date_col].dt.year
    yearly = df.groupby("year")[value_col].sum().reset_index()

    if len(yearly) < 4:
        return None

    result = mk.original_test(yearly[value_col].values)
    return {"slope": result.slope, "p": result.p, "trend": result.trend}


well_trends = []

for well_id in all_event_tables.keys():
    table = add_covariates(all_event_tables[well_id])

    precip_trend = compute_well_trend(table, "MAG")
    recharge_trend = compute_well_trend(table, "RECH")

    if precip_trend is None or recharge_trend is None:
        continue

    lat = df_ne_wells.loc[df_ne_wells["usgs_id"].astype(str) == well_id, "lat"].values
    lon = df_ne_wells.loc[df_ne_wells["usgs_id"].astype(str) == well_id, "lon"].values

    if len(lat) == 0:
        continue

    well_trends.append({
        "well_id": well_id,
        "lat": lat[0],
        "lon": lon[0],
        "precip_slope": precip_trend["slope"],
        "precip_p": precip_trend["p"],
        "recharge_slope": recharge_trend["slope"],
        "recharge_p": recharge_trend["p"],
    })

well_trends_df = pd.DataFrame(well_trends)
print(f"Wells with computable trends: {len(well_trends_df)}")
print(well_trends_df.head())

Wells with computable trends: 163
           well_id        lat        lon  precip_slope  precip_p  \
0  390211074505502  39.036502 -74.848224     -0.406000  1.000000   
1  391145074520401  39.195948 -74.867392      5.303333  0.397587   
2  391621074435401  39.272615 -74.731274     13.780000  0.372691   
3  392232074234403  39.395672 -74.625158     10.470000  0.246387   
4  392731075092401  39.459004 -75.157683      5.312800  0.186426   

   recharge_slope  recharge_p  
0        0.019348    0.766525  
1        0.022939    0.310046  
2        0.189474    0.192616  
3        0.100176    0.099509  
4        0.035847    0.018452  


In [34]:
def plot_trend_map(well_trends_df, value_col, title, states_shp=state_boundaries):
    states = gpd.read_file(states_shp)
    ne = states[states["NAME"].isin(NE_states)].copy()
    ne["geometry"] = ne["geometry"].buffer(0)
    ne = ne.set_crs(epsg=4326, allow_override=True)

    fig, ax = plt.subplots(figsize=(10, 10))
    ne.boundary.plot(ax=ax, color="black", linewidth=0.6)

    vmax = well_trends_df[value_col].abs().max()
    scatter = ax.scatter(
        well_trends_df["lon"], well_trends_df["lat"],
        c=well_trends_df[value_col], cmap="RdBu",
        vmin=-vmax, vmax=vmax,
        s=80, edgecolor="black", linewidth=0.5, zorder=3
    )

    cbar = fig.colorbar(scatter, ax=ax, shrink=0.7)
    cbar.set_label(f"{value_col} (mm/yr)")

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title)
    fig.tight_layout()
    plt.show()


plot_trend_map(well_trends_df, "precip_slope", "Precipitation Trend by Well (mm/yr)")

In [35]:
plot_trend_map(well_trends_df, "recharge_slope", "Recharge Trend by Well (mm/yr)")